# CRC Visium HD — end-to-end ASTER spatial domains (Fig. 4a)

Runs the whole colorectal cancer workflow from the raw Visium HD bundle and the
standardized H&E to the K=15 spatial domain map of Fig. 4a (545,613 in-tissue
8 µm bins x 2500 spatially variable genes).

Every model and training loop comes from `repro_st_aster`; this notebook only
wires the stages together. The equivalent command-line path is
`scripts/prepare_crc_visiumhd_*.sh` + `scripts/reproduce_crc_visiumhd_*.sh`.

## Data you need to download first

This repository ships the data directories **empty**. Download the bundle and
unpack it into the paths below (see `raw_data/crc_visiumhd/README.md`):

```text
# 1. Raw Visium HD bundle (square_008um) -> raw_data/crc_visiumhd/binned_outputs/
#    <CRC_RAW_ST_URL>
# 2. Standardized H&E image + metadata.json -> raw_data/crc_visiumhd/
#    <CRC_HE_URL>
# 3. Pre-extracted UNI-2 features -> preprocess_data/crc_visiumhd/uni/
#    <CRC_UNI_FEATURES_URL>
```

Item 3 is optional if you have your own UNI-2 weights: set `HAVE_UNI2_WEIGHTS = True`
below and the features are extracted on the fly instead. UNI-2 weights are not
redistributed here.

Environment: `conda activate repro_st_aster` (see the top-level README).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from repro_st_aster.common import find_repo_root, knn_gaussian_smooth
from repro_st_aster.uni_bcam.prepare_inputs_visiumhd import prepare_inputs_hd

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'raw_data' / 'crc_visiumhd'
PRE_DIR = REPO_ROOT / 'preprocess_data' / 'crc_visiumhd'
UNI_DIR = PRE_DIR / 'uni'
BCAM_INPUT_DIR = PRE_DIR / 'bcam_input'
INR_DIR = PRE_DIR / 'inr_output'
FUSION_DIR = PRE_DIR / 'bcam_output'
VIS_DIR = PRE_DIR / 'viz'

BIN_DIR = RAW_DIR / 'binned_outputs' / 'square_008um'
MTX_DIR = BIN_DIR / 'filtered_feature_bc_matrix'
POSITIONS_PARQUET = BIN_DIR / 'spatial' / 'tissue_positions.parquet'
HE_METADATA = RAW_DIR / 'metadata.json'
HE_IMAGE = RAW_DIR / 'tissue_standardized_0p5um.jpg'

# Route A extracts UNI-2 features from the H&E (needs the gated weights);
# route B loads the pre-extracted feature grid. Everything downstream is identical.
HAVE_UNI2_WEIGHTS = False
UNI2_WEIGHTS_DIR = RAW_DIR / 'uni2-h'

# Set an integer (e.g. 20000) for a quick smoke run; None reproduces the full figure.
# This is the notebook equivalent of the CLI --max-cells flag.
SUBSET_N = None

for path in [MTX_DIR, POSITIONS_PARQUET, HE_METADATA]:
    print(f'{path.exists()!s:>5}  {path}')

## Step 0 — raw bins to model inputs

`prepare_inputs_hd` reads the MTX bundle and `tissue_positions.parquet`, keeps the
in-tissue bins, maps their full-resolution pixel coordinates into standardized H&E
space using `metadata.json`, drops mitochondrial / ribosomal / housekeeping genes,
normalizes (`normalize_total(1e4)` + `log1p`), selects the top-2500 spatially
variable genes by Moran's I, and finally keeps the bins that fall inside the
standardized H&E image — the bins that have a UNI-2 superpixel.

That last filter is what takes 545,913 in-tissue bins to the **545,613** the
published run uses; the 300 dropped bins all sit above the image's top edge.

In [ ]:
meta = prepare_inputs_hd(
    mtx_dir=MTX_DIR,
    positions_parquet=POSITIONS_PARQUET,
    he_metadata=HE_METADATA,
    uni_feature_path=UNI_DIR / 'superpixel_features.npy',
    out_dir=BCAM_INPUT_DIR,
    top_svg=2500,
    svg_k=15,
    svg_subsample=12000,
    superpixel_stride=14,   # colorectal step; matches the UNI-2 grid stride
    max_cells=SUBSET_N,
    seed=42,
)
for key, value in meta.items():
    print(f'  {key:26s} {value}')

## Step 1 — UNI-2 histology features

The standardized H&E is cut into 224x224 tiles; UNI-2 (ViT-Giant/14) emits 16x16
patch tokens per tile, stitched into a `(912, 1008, 1536)` superpixel grid where one
superpixel spans 14 px (7 µm).

Route B just checks that the downloaded grid is present — Step 0 already read it to
attach a feature vector to every bin, so nothing else is needed here.

In [ ]:
if HAVE_UNI2_WEIGHTS:
    from repro_st_aster.uni_bcam.uni_extract import extract_features

    uni_meta = extract_features(
        image_path=HE_IMAGE,
        model_dir=UNI2_WEIGHTS_DIR,
        out_dir=UNI_DIR,
        superpixel_stride=14,
        write_pickle=False,
    )
    print(uni_meta)
else:
    uni_grid = np.load(UNI_DIR / 'superpixel_features.npy', mmap_mode='r')
    print('pre-extracted UNI-2 grid:', uni_grid.shape)

uni_per_bin = np.load(BCAM_INPUT_DIR / 'uni2_features_per_cell.npy', mmap_mode='r')
print('UNI-2 features per bin:  ', uni_per_bin.shape)

## Step 2 — INR + low-rank Tucker-2 reconstruction

`reconstruct_hd` fits the continuous field `f(s, g) = SIREN(s) @ K @ g_emb^T`
(depth 8, spatial rank 512, gene rank 256) over an 8000-epoch schedule with an
omega ramp of 1 -> 10, training on **all** bins with no validation split. Each
batch balances zeros against non-zeros to a 30% non-zero ratio and optimises a
weighted MSE (non-zeros weighted 3x), which keeps the sparse signal from being
washed out.

The published reconstruction is the **epoch-4000 snapshot**, so `--recon-epoch 4000`
emits that snapshot and stops. Full runs take hours on one GPU; use `SUBSET_N` plus a
small `--epochs` for a smoke test.

In [ ]:
from repro_st_aster.aster_sc import reconstruct_hd

# Study-run schedule: 8000 epochs, reconstruction taken from the epoch-4000 snapshot.
inr_args = [
    '--data-dir', str(BCAM_INPUT_DIR),
    '--out-dir', str(INR_DIR),
    '--depth', '8',
    '--batch-spots', '32768',
    '--epochs', '20' if SUBSET_N else '8000',
    '--recon-epoch', '20' if SUBSET_N else '4000',
    '--snapshot-every', '10' if SUBSET_N else '500',
]
if SUBSET_N:
    inr_args += ['--max-cells', str(SUBSET_N)]

reconstruct_hd.main(inr_args)

## Step 3 — FusionNet: INR expression x UNI-2 morphology

`FusionNet` (in `repro_st_aster.aster_sc.fusion_core`) runs two stacked bidirectional
cross-attention blocks over each bin's K=8 neighbourhood — **including the bin
itself as neighbour 0** — concatenates the centre token with the centre bin's raw
UNI-2 vector, and reconstructs the raw lognorm expression through a 1792 -> 1024 ->
512 -> 512 -> G MLP (smooth-L1 loss, 30 epochs, lr 1e-4).

The clustering latent is the 512-d activation *before* the final linear layer, so the
gene reconstruction head stays intact. Note the return order is `(pred, latent)`,
the reverse of `BCAM`.

In [ ]:
from repro_st_aster.aster_sc import fusion

fusion_args = [
    '--data-dir', str(BCAM_INPUT_DIR),
    '--inr-dir', str(INR_DIR),
    '--out-dir', str(FUSION_DIR),
    '--epochs', '2' if SUBSET_N else '30',
    '--batch-size', '256',
    '--k-neighbors', '8',
    '--embed-dim', '256',
    '--num-heads', '8',
]
if SUBSET_N:
    fusion_args += ['--max-cells', str(SUBSET_N)]

fusion.main(fusion_args)

## Step 4 — spatial domains (KMeans K=15)

The latent is first smoothed over its 30 nearest neighbours with a Gaussian kernel
(sigma = mean neighbour distance), which is what turns a per-bin embedding into
spatially coherent regions. `KMeans(K=15, random_state=0, n_init='auto')` on the
smoothed latent gives the Fig. 4a domains.

Fig. 4a colours domains with matplotlib's `tab20` applied directly to the label
array, so `--palette none` (the default) is the published appearance.

In [ ]:
from repro_st_aster.aster_sc import cluster_visualize

cluster_args = [
    '--data-dir', str(BCAM_INPUT_DIR),
    '--bcam-dir', str(FUSION_DIR),
    '--inr-dir', str(INR_DIR),
    '--vis-dir', str(VIS_DIR),
    '--k', '15',
    '--smooth-k', '30',
    '--kmeans-random-state', '0',
    '--kmeans-n-init', 'auto',
    '--palette', 'none',
    '--point-size', '0.7', '--point-alpha', '0.9', '--marker', 's', '--invert-yaxis',
    '--extra-labels-name', 'kmeans_labels_K15.npy',
]
if SUBSET_N:
    cluster_args += ['--max-cells', str(SUBSET_N)]

cluster_visualize.main(cluster_args)

In [ ]:
labels = np.load(VIS_DIR / 'kmeans_labels_K15.npy')
coords = np.load(BCAM_INPUT_DIR / 'cell_coords_standardized.npy')[: len(labels)]

span = np.ptp(coords, axis=0)
fig, ax = plt.subplots(figsize=(9 * span[0] / span[1], 9))
ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap='tab20', s=0.7,
           marker='s', alpha=0.9, edgecolors='none', rasterized=True)
ax.set_title(f'CRC ASTER spatial domains (K=15, {len(labels):,} bins)')
ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
plt.tight_layout()

for domain_id, count in zip(*np.unique(labels, return_counts=True)):
    print(f'  domain {domain_id:2d}: {count:7,d} bins ({count / len(labels) * 100:5.2f}%)')

## Rerunning from the shell

The same pipeline without a notebook:

```bash
bash scripts/prepare_crc_visiumhd_uni.sh --model-dir /path/to/uni2-h   # optional, route A only
bash scripts/prepare_crc_visiumhd_bcam_input.sh
bash scripts/reproduce_crc_visiumhd_inr.sh
bash scripts/reproduce_crc_visiumhd_fusion.sh
bash scripts/reproduce_crc_visiumhd_cluster.sh
```

Smoke test (minutes rather than hours):

```bash
bash scripts/prepare_crc_visiumhd_bcam_input.sh --top-svg 500 --svg-subsample 5000 --max-cells 20000
bash scripts/reproduce_crc_visiumhd_inr.sh --epochs 20 --recon-epoch 20 --snapshot-every 10 --max-cells 20000
bash scripts/reproduce_crc_visiumhd_fusion.sh --epochs 2 --max-cells 20000
bash scripts/reproduce_crc_visiumhd_cluster.sh --max-cells 20000
```